# Notebook para descarga de Iinformación de AIS

## 0. Configuración inicial

In [0]:
CATALOGO = "workspace"
ESQUEMA = "oceanwatch"

RUTA_LANDING = (
    "/Volumes/workspace/oceanwatch/"
    "ais_landing/incoming/"
)

print("CONFIGURACIÓN DE DESCARGA AIS")
print("=" * 60)

# Verificar acceso a Landing
archivos = dbutils.fs.ls(RUTA_LANDING)

print(f"Catálogo: {CATALOGO}")
print(f"Esquema: {ESQUEMA}")
print(f"Ruta Landing: {RUTA_LANDING}")
print(f"Elementos existentes: {len(archivos)}")

print("\n[OK] Carpeta Landing disponible")
print("[OK] Configuración inicial completada")


CONFIGURACIÓN DE DESCARGA AIS
Catálogo: workspace
Esquema: oceanwatch
Ruta Landing: /Volumes/workspace/oceanwatch/ais_landing/incoming/
Elementos existentes: 0

[OK] Carpeta Landing disponible
[OK] Configuración inicial completada


## 1. Configuación control de descargas

In [0]:
import json
import os
from datetime import datetime, timezone

# Manifiesto fuera de la carpeta monitoreada por Auto Loader
RUTA_MANIFIESTO = (
    "/Volumes/workspace/oceanwatch/"
    "ais_landing/control/descargas_ais.json"
)

os.makedirs(os.path.dirname(RUTA_MANIFIESTO), exist_ok=True)


def cargar_manifiesto():
    if not os.path.exists(RUTA_MANIFIESTO):
        return {}

    with open(RUTA_MANIFIESTO, "r", encoding="utf-8") as f:
        return json.load(f)


def verificar_descarga(url):
    manifiesto = cargar_manifiesto()
    registro = manifiesto.get(url)

    if registro and os.path.isfile(registro["ruta_local"]):
        print("[OMITIDO] Archivo previamente descargado")
        print(f"URL: {url}")
        print(f"Archivo: {registro['ruta_local']}")
        print(f"Fecha de descarga: {registro['fecha_descarga']}")
        return True

    return False


def registrar_descarga(url, ruta_local):
    manifiesto = cargar_manifiesto()

    fecha_descarga = datetime.now(
        timezone.utc
    ).isoformat(timespec="seconds")

    manifiesto[url] = {
        "ruta_local": ruta_local,
        "fecha_descarga": fecha_descarga,
        "estado": "COMPLETADO"
    }

    ruta_temporal = RUTA_MANIFIESTO + ".tmp"

    with open(ruta_temporal, "w", encoding="utf-8") as f:
        json.dump(manifiesto, f, indent=4, ensure_ascii=False)

    os.replace(ruta_temporal, RUTA_MANIFIESTO)

    print(f"[OK] Descarga registrada: {fecha_descarga}")


print("[OK] Control de duplicados configurado")
print(f"Registros existentes: {len(cargar_manifiesto())}")
print("[OK] No se han descargado archivos")


[OK] Control de duplicados configurado
Registros existentes: 0
[OK] No se han descargado archivos


## 2. Descarga de información AIS

In [0]:
import os
import requests
import zipfile
import shutil
import tempfile

BASE_URL = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023"

archivos = [
    f"AIS_2023_06_{dia:02d}.zip"
    for dia in range(1, 8)
]

urls = [f"{BASE_URL}/{archivo}" for archivo in archivos]

RUTA_ZIP = (
    "/Volumes/workspace/oceanwatch/"
    "ais_landing/originales/"
)

os.makedirs(RUTA_ZIP, exist_ok=True)
os.makedirs(RUTA_LANDING, exist_ok=True)

for url in urls:

    nombre_zip = url.rsplit("/", 1)[-1]
    ruta_zip = os.path.join(RUTA_ZIP, nombre_zip)

    print(f"\nARCHIVO: {nombre_zip}")
    print("-" * 60)

    # Omitir archivos previamente descargados
    if verificar_descarga(url):
        continue

    try:
        with tempfile.TemporaryDirectory(dir=RUTA_ZIP) as temporal:

            ruta_temporal = os.path.join(temporal, nombre_zip)

            # Descargar ZIP por fragmentos
            with requests.get(
                url,
                stream=True,
                timeout=(30, 180)
            ) as respuesta:

                respuesta.raise_for_status()

                with open(ruta_temporal, "wb") as f:
                    for bloque in respuesta.iter_content(
                        chunk_size=8 * 1024 * 1024
                    ):
                        if bloque:
                            f.write(bloque)

            # Verificar integridad del ZIP
            with zipfile.ZipFile(ruta_temporal) as z:
                if z.testzip() is not None:
                    raise RuntimeError("ZIP corrupto")

                csv_incluidos = [
                    nombre for nombre in z.namelist()
                    if nombre.lower().endswith(".csv")
                    and not nombre.endswith("/")
                ]

                if not csv_incluidos:
                    raise RuntimeError("ZIP sin archivos CSV")

                # Publicar los CSV completos en Landing
                for nombre_csv in csv_incluidos:

                    destino = os.path.join(
                        RUTA_LANDING,
                        os.path.basename(nombre_csv)
                    )

                    if os.path.exists(destino):
                        raise RuntimeError(
                            f"Ya existe un CSV en Landing: {destino}"
                        )

                    with z.open(nombre_csv) as origen:
                        with tempfile.NamedTemporaryFile(
                            dir=RUTA_LANDING,
                            delete=False,
                            suffix=".tmp"
                        ) as archivo_temporal:

                            shutil.copyfileobj(
                                origen,
                                archivo_temporal
                            )
                            ruta_csv_temporal = archivo_temporal.name

                    os.replace(ruta_csv_temporal, destino)

            # Conservar ZIP original
            os.replace(ruta_temporal, ruta_zip)

        # Registrar únicamente la descarga completada
        registrar_descarga(url, ruta_zip)

        print(f"[OK] ZIP descargado: {nombre_zip}")
        print(f"[OK] CSV publicados: {len(csv_incluidos)}")

    except Exception as e:
        print(f"[ERROR] {nombre_zip}: {e}")
        raise

print("\n[OK] Proceso de descarga finalizado")



ARCHIVO: AIS_2023_06_01.zip
------------------------------------------------------------
[OMITIDO] Archivo previamente descargado
URL: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_01.zip
Archivo: /Volumes/workspace/oceanwatch/ais_landing/originales/AIS_2023_06_01.zip
Fecha de descarga: 2026-11-22T16:46:28+00:00

ARCHIVO: AIS_2023_06_02.zip
------------------------------------------------------------
[OK] Descarga registrada: 2026-11-22T17:07:07+00:00
[OK] ZIP descargado: AIS_2023_06_02.zip
[OK] CSV publicados: 1

ARCHIVO: AIS_2023_06_03.zip
------------------------------------------------------------
[OK] Descarga registrada: 2026-11-22T17:07:39+00:00
[OK] ZIP descargado: AIS_2023_06_03.zip
[OK] CSV publicados: 1

ARCHIVO: AIS_2023_06_04.zip
------------------------------------------------------------
[OK] Descarga registrada: 2026-11-22T17:08:01+00:00
[OK] ZIP descargado: AIS_2023_06_04.zip
[OK] CSV publicados: 1

ARCHIVO: AIS_2023_06_05.zip
--------------------

## 3. Validación de Landing

In [0]:
import os

print("VALIDACIÓN DE ARCHIVOS AIS")
print("=" * 65)

directorios = {
    "ZIP originales": RUTA_ZIP,
    "CSV Landing": RUTA_LANDING
}

for nombre, ruta in directorios.items():

    archivos = [
        f for f in os.listdir(ruta)
        if os.path.isfile(os.path.join(ruta, f))
    ]

    print(f"\n{nombre}")
    print("-" * 65)

    for archivo in sorted(archivos):
        ruta_archivo = os.path.join(ruta, archivo)
        tamano_mb = os.path.getsize(ruta_archivo) / (1024 ** 2)

        print(f"{archivo} | {tamano_mb:,.2f} MB")

    print(f"Total archivos: {len(archivos)}")

    temporales = [
        f for f in archivos
        if f.endswith(".tmp")
    ]

    if temporales:
        raise RuntimeError(
            f"Se encontraron archivos temporales en {ruta}"
        )

print("\n[OK] Validación de directorios completada")


VALIDACIÓN DE ARCHIVOS AIS

ZIP originales
-----------------------------------------------------------------
AIS_2023_06_01.zip | 328.02 MB
AIS_2023_06_02.zip | 337.74 MB
AIS_2023_06_03.zip | 297.78 MB
AIS_2023_06_04.zip | 317.06 MB
AIS_2023_06_05.zip | 320.11 MB
AIS_2023_06_06.zip | 319.77 MB
AIS_2023_06_07.zip | 331.80 MB
Total archivos: 7

CSV Landing
-----------------------------------------------------------------
AIS_2023_06_01.csv | 899.73 MB
AIS_2023_06_02.csv | 924.53 MB
AIS_2023_06_03.csv | 820.65 MB
AIS_2023_06_04.csv | 870.98 MB
AIS_2023_06_05.csv | 880.06 MB
AIS_2023_06_06.csv | 876.26 MB
AIS_2023_06_07.csv | 909.44 MB
Total archivos: 7

[OK] Validación de directorios completada


## 4. Validación de estructura CSV

In [0]:
import os
import csv

print("VALIDACIÓN DE ESTRUCTURA CSV")
print("=" * 65)

archivos_csv = sorted([
    archivo
    for archivo in os.listdir(RUTA_LANDING)
    if archivo.lower().endswith(".csv")
])

esquema_referencia = None

for archivo in archivos_csv:

    ruta = os.path.join(RUTA_LANDING, archivo)

    with open(ruta, "r", encoding="utf-8-sig", newline="") as f:
        encabezado = next(csv.reader(f, delimiter=","))

    if esquema_referencia is None:
        esquema_referencia = encabezado

    compatible = encabezado == esquema_referencia

    print(
        f"{archivo} | "
        f"Columnas: {len(encabezado)} | "
        f"Esquema compatible: {compatible}"
    )

    if not compatible:
        raise RuntimeError(
            f"Estructura diferente en {archivo}"
        )

print("\nCOLUMNAS IDENTIFICADAS")
print("-" * 65)

for numero, columna in enumerate(esquema_referencia, 1):
    print(f"{numero:02d}. {columna}")

print("\n[OK] Encabezados CSV consistentes")


VALIDACIÓN DE ESTRUCTURA CSV
AIS_2023_06_01.csv | Columnas: 17 | Esquema compatible: True
AIS_2023_06_02.csv | Columnas: 17 | Esquema compatible: True
AIS_2023_06_03.csv | Columnas: 17 | Esquema compatible: True
AIS_2023_06_04.csv | Columnas: 17 | Esquema compatible: True
AIS_2023_06_05.csv | Columnas: 17 | Esquema compatible: True
AIS_2023_06_06.csv | Columnas: 17 | Esquema compatible: True
AIS_2023_06_07.csv | Columnas: 17 | Esquema compatible: True

COLUMNAS IDENTIFICADAS
-----------------------------------------------------------------
01. MMSI
02. BaseDateTime
03. LAT
04. LON
05. SOG
06. COG
07. Heading
08. VesselName
09. IMO
10. CallSign
11. VesselType
12. Status
13. Length
14. Width
15. Draft
16. Cargo
17. TransceiverClass

[OK] Encabezados CSV consistentes


In [0]:

# Verificar existencia del esquema OceanWatch

spark.sql("SHOW SCHEMAS IN workspace").show(
    truncate=False
)


+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
|oceanwatch        |
+------------------+

